## VCP — Volatility Contraction Pattern Strategy

A pure-technical, **long-only** breakout strategy on a broad US-equity universe
(S&P-500-constituent PERMNOs, CRSP schema).

**Pipeline**
1. **Data** — daily OHLCV per PERMNO.
2. **Pivots** — Directional Change with *adaptive* (ATR-based, time-varying) sigma, so
   each historical pattern is detected with the volatility regime present at that date
   (no future-ATR leakage).
3. **Patterns** — every rising-structure VCP instance across full history.
4. **Trade construction** — buy-signal / breakout entries, fixed-fractional sizing.
5. **Backtest** — portfolio-aware `BacktestEngine` via the `VCPStrategy` adapter.

In [62]:
import sys
sys.path.insert(0, '../../../..')
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [63]:
import pandas as pd
import numpy as np
from math import isfinite
from typing import Optional

SNP500_CSV_PATH = r"C:\Users\bob.liang\Downloads\snp500_index_data.csv"

def load_snp500_data(csv_path: str, start_date: Optional[str] = None) -> pd.DataFrame:
    dtype_map = {
        "PERMNO":   "Int64",
        "Ticker":   str,
        "DlyCalDt": str,
        "DlyOpen":  float,
        "DlyHigh":  float,
        "DlyLow":   float,
        "DlyClose": float,
        "DlyVol":   float,
        "DlyCumFacPr": float,
        "DlyCumFacShr": float,
    }
    raw = pd.read_csv(csv_path, usecols=list(dtype_map.keys()) + ["MbrStartDt", "MbrEndDt"],
                      dtype=dtype_map, low_memory=False)
    raw["Date"]       = pd.to_datetime(raw["DlyCalDt"],  errors="coerce")
    raw["MbrStartDt"] = pd.to_datetime(raw["MbrStartDt"], errors="coerce")
    raw["MbrEndDt"]   = pd.to_datetime(raw["MbrEndDt"],   errors="coerce")
    raw = raw.dropna(subset=["Date", "PERMNO"])
    if start_date is not None:
        raw = raw[raw['Date'] >= pd.Timestamp(start_date)]
    last_ticker = raw.sort_values("Date").groupby("PERMNO")["Ticker"].last().astype(str)
    raw["InstrumentKey"] = "PERMNO_" + raw["PERMNO"].astype(str) + "_" + raw["PERMNO"].map(last_ticker)
    raw = raw.rename(columns={"DlyOpen":"Open","DlyHigh":"High","DlyLow":"Low","DlyClose":"Close","DlyVol":"Volume"})
    fac = pd.to_numeric(raw["DlyCumFacPr"], errors="coerce").where(lambda x: x > 0).fillna(1.0)
    for _c in ["Open","High","Low","Close"]: raw[_c] = raw[_c] / fac
    raw["Volume"] = raw["Volume"] * fac
    for col in ["Open","High","Low","Close"]: raw[col] = raw[col].where(raw[col] > 0, other=np.nan)
    raw["Volume"] = raw["Volume"].where(raw["Volume"] >= 0, other=np.nan)
    wide = raw.pivot_table(index="Date", columns="InstrumentKey",
                           values=["Open","High","Low","Close","Volume"], aggfunc="last")
    wide.columns.set_names(["field", "Ticker"], inplace=True)
    wide.index = pd.DatetimeIndex(wide.index); wide = wide.sort_index()
    instruments = wide.columns.get_level_values("Ticker").unique()
    print(f"Loaded S&P 500 constituent data")
    print(f"  Instruments: {len(instruments):,}   Date range: {wide.index[0].date()} → {wide.index[-1].date()}")
    print(f"  Rows: {len(wide):,}")
    return wide

BT_START_DATE = "2020-01-01"
data = load_snp500_data(SNP500_CSV_PATH, start_date=BT_START_DATE)
tmp_data = data.copy()

Loaded S&P 500 constituent data
  Instruments: 594   Date range: 2020-01-02 → 2024-12-31
  Rows: 1,258


## Configuration

In [64]:
from datetime import datetime
end_date = datetime.today()
dc_sigma_warmup    = 0.07
dc_atr_multiplier  = 1.5
dc_sigma_floor     = 0.04
dc_sigma_ceiling   = 0.20
dc_atr_window      = 20
last_contraction_pct = 0.12
breakout_window       = 20
breakout_buffer       = 0.003
vol_surge_window      = 20
vol_surge_multiplier  = 1.5
vol_dryup_window      = 10

In [65]:
def prepare_ohlc(df):
    df = df.copy(); df.columns = [col.lower() for col in df.columns]
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    df = df.sort_index(); df = df[~df.index.duplicated(keep='first')]; df = df.ffill(); return df

## Pivot Detection

In [66]:
def directional_change(close, high, low, sigma_or_series):
    scalar_sigma = np.isscalar(sigma_or_series)
    if not scalar_sigma:
        sigma_arr = np.asarray(sigma_or_series, dtype=float).copy()
        _last = dc_sigma_warmup
        for k in range(len(sigma_arr)):
            if np.isnan(sigma_arr[k]) or sigma_arr[k] <= 0: sigma_arr[k] = _last
            else: _last = sigma_arr[k]
    up_zig = True
    tmp_max = high[0]; tmp_min = low[0]; tmp_max_i = 0; tmp_min_i = 0
    tops = []; bottoms = []
    for i in range(len(close)):
        sigma = float(sigma_or_series) if scalar_sigma else float(np.clip(sigma_arr[i], dc_sigma_floor, dc_sigma_ceiling))
        if up_zig:
            if high[i] > tmp_max: tmp_max = high[i]; tmp_max_i = i
            elif close[i] < tmp_max * (1 - sigma):
                tops.append([i, tmp_max_i, tmp_max]); up_zig = False; tmp_min = low[i]; tmp_min_i = i
        else:
            if low[i] < tmp_min: tmp_min = low[i]; tmp_min_i = i
            elif close[i] > tmp_min * (1 + sigma):
                bottoms.append([i, tmp_min_i, tmp_min]); up_zig = True; tmp_max = high[i]; tmp_max_i = i
    return tops, bottoms

In [67]:
def compute_atr(df, window):
    high=df['high']; low=df['low']; close=df['close']
    tr = pd.DataFrame({'h-l':high-low,'h-pc':(high-close.shift(1)).abs(),'l-pc':(low-close.shift(1)).abs()}).max(axis=1)
    return tr.rolling(window).mean()

## Pattern Detection

In [68]:
def detect_all_rising_structures(tops, bottoms, min_pts=3, tolerance=0):
    patterns = []
    n = min(len(tops), len(bottoms))
    for i in range(n - min_pts + 1):
        top_slice = tops[i:i+min_pts]; bot_slice = bottoms[i:i+min_pts]
        if len(top_slice)!=min_pts or len(bot_slice)!=min_pts: continue
        hp = [t[2] for t in top_slice]; lp = [b[2] for b in bot_slice]
        hv = sum(1 for j in range(min_pts-1) if hp[j] >= hp[j+1])
        lv = sum(1 for j in range(min_pts-1) if lp[j] >= lp[j+1])
        if hv <= tolerance and lv <= tolerance:
            patterns.append({'start_idx':min(top_slice[0][1],bot_slice[0][1]),
                             'end_idx':max(top_slice[-1][1],bot_slice[-1][1]),
                             'top_points':top_slice,'bottom_points':bot_slice})
    return patterns

In [69]:
def extract_pattern_metadata(pattern, df_stock):
    tp=pattern['top_points']; bp=pattern['bottom_points']; start_idx=pattern['start_idx']
    last_high=tp[-1][2]; last_low=bp[-1][2]
    last_swing_pct=(last_high-last_low)/last_high if last_high>0 else 1.0
    buy_signal_idx=buy_signal_date=buy_signal_price=None
    if last_swing_pct <= last_contraction_pct:
        buy_signal_idx = max(tp[-1][0], bp[-1][0])
        buy_signal_date = str(df_stock.index[buy_signal_idx].date())
        buy_signal_price = float(df_stock['close'].iloc[buy_signal_idx])
    return {'buy_signal_idx':buy_signal_idx,'buy_signal_date':buy_signal_date,
            'buy_signal_price':buy_signal_price,'last_swing_pct':last_swing_pct,
            'resistance':last_high,'chart_start_idx':start_idx}

## Multi-Ticker VCP Scanner (Rolling, No Look-Ahead)

In [70]:
results = []
all_tickers = data.columns.get_level_values('Ticker').unique()
rolling_warmup = dc_atr_window + 30
for ticker in all_tickers:
    df_raw = data.xs(ticker, level='Ticker', axis=1); df_full = prepare_ohlc(df_raw)
    if len(df_full) < 50: continue
    close_np=df_full['close'].to_numpy(); high_np=df_full['high'].to_numpy()
    low_np=df_full['low'].to_numpy(); volume_np=df_full['volume'].to_numpy()
    atr_series = compute_atr(df_full, dc_atr_window)
    sigma_series = (dc_atr_multiplier * atr_series / df_full['close']).clip(dc_sigma_floor, dc_sigma_ceiling).to_numpy()
    tops, bottoms = directional_change(close_np, high_np, low_np, sigma_series)
    recorded_ends: set[int] = set()
    for bar_t in range(rolling_warmup, len(df_full)):
        tops_t = [t for t in tops if t[0] <= bar_t]
        bots_t = [b for b in bottoms if b[0] <= bar_t]
        if len(tops_t) < 3 or len(bots_t) < 3: continue
        all_patterns = detect_all_rising_structures(tops_t, bots_t)
        if not all_patterns: continue
        latest = all_patterns[-1]; end_idx = latest['end_idx']
        tp = latest['top_points']; bp = latest['bottom_points']
        last_high = tp[-1][2]; last_low = bp[-1][2]; resistance = last_high
        last_swing_pct = (last_high-last_low)/last_high if last_high>0 else 1.0
        if last_swing_pct > last_contraction_pct: continue
        if end_idx in recorded_ends: continue
        scan_end = min(len(df_full), end_idx+1+breakout_window)
        invalidated = any(close_np[bar] < last_low for bar in range(end_idx+1, min(bar_t+1, scan_end)))
        if invalidated: continue
        recorded_ends.add(end_idx)
        confirm_idx = max(tp[-1][0], bp[-1][0])
        breakout_idx = None; breakout_date = "N/A"
        vol_base_start = max(0, end_idx - vol_surge_window)
        vol_baseline = float(df_full['volume'].iloc[vol_base_start:end_idx].mean())
        for bar in range(end_idx+1, min(len(df_full), end_idx+1+breakout_window)):
            if bar > bar_t: break
            if close_np[bar] > resistance*(1+breakout_buffer) and \
               (volume_np[bar] > vol_surge_multiplier*vol_baseline if pd.notna(vol_baseline) and vol_baseline>0 else False):
                breakout_idx=bar; breakout_date=str(df_full.index[bar].date()); break
        chart_start_idx = min(tp[0][1], bp[0][1])
        results.append({
            '_ticker':ticker,'_df_full':df_full,'_atr_series':atr_series,
            '_tops_full':tops,'_bottoms_full':bottoms,'_tops':tops_t,'_bottoms':bots_t,
            '_pattern':latest,
            '_meta':{'buy_signal_idx':confirm_idx,'buy_signal_date':str(df_full.index[confirm_idx].date()),
                     'buy_signal_price':float(df_full['close'].iloc[confirm_idx]),
                     'last_swing_pct':last_swing_pct,'resistance':resistance,'chart_start_idx':chart_start_idx},
            '_breakout_idx':breakout_idx,'_chart_start_idx':chart_start_idx,
            '_chart_end_idx':min(len(df_full)-1, end_idx+breakout_window),
            'Ticker':ticker,'PatternStart':str(df_full.index[chart_start_idx].date()),
            'PatternEnd':str(df_full.index[end_idx].date()),
            'BuySignalDate':str(df_full.index[confirm_idx].date()),
            'LastSwingPct':round(last_swing_pct*100,2),'BreakoutDate':breakout_date})
results_df = pd.DataFrame(results).sort_values('BuySignalDate').reset_index(drop=True)
results_df.index.name = 'VCP_ID'; results_df = results_df.reset_index()
print(f"Total causal VCP buy signals detected: {len(results_df)}")
if len(results_df) > 0:
    print(f"  Date range: {results_df['BuySignalDate'].min()} → {results_df['BuySignalDate'].max()}  |  {results_df['Ticker'].nunique()} instruments")

Total causal VCP buy signals detected: 3547
  Date range: 2020-04-24 → 2024-12-26  |  481 instruments


## Backtesting Layer

In [71]:
from datetime import timedelta
from collections import defaultdict
from copy import deepcopy
from src.backtest.engine import BacktestEngine
from src.backtest.strategies import Strategy, EqualWeighted
from src.models.backtest_model import Position, Trade

class BuyAndHold(Strategy):
    def __init__(self,name,tickers): self._name=name; self.tickers=tickers
    @property
    def name(self): return self._name
    def engine_start(self): pass
    def on_bar(self):
        if self.current_bar_idx==self.price_df.index[0]:
            self.execute_trades_by_weights({t:1.0/len(self.tickers) for t in self.tickers})

In [72]:
BT = dict(initial_cash=1_000_000.0,risk_free_rate=0.04,margin_spread=0.01,
    entry_mode='buysignal_preferred',buysignal_entry='confirm',buysignal_entry_lag=1,
    target_mode='min_r_or_resistance',R_target=5.0,partial_frac=0.33,trail_pct=0.15,
    max_hold_bars=120,slippage_pct=0.001,commission_pct=0.0005,
    risk_per_trade_pct=0.01,max_concurrent_positions=15,
    breakout_close_in_upper_pct=None,breakout_volume_mult=None,breakout_atr_expansion_min=None,
    wf_is_ratio=0.70,wf_random_seed=42)

In [73]:
import hashlib
def _find_next_resistance_fwd(df_full, entry_idx, resistance):
    for i in range(entry_idx+1, len(df_full)):
        if df_full['high'].iloc[i] > resistance: return float(df_full['high'].iloc[i])
    return None
def _assign_wf_buckets(results_df, BT):
    unique_tickers = results_df['Ticker'].unique()
    def _bucket(t):
        h = int(hashlib.md5(t.encode()).hexdigest(), 16)
        return 'IS' if np.random.default_rng(h^BT['wf_random_seed']).random() < BT['wf_is_ratio'] else 'OOS'
    return results_df['Ticker'].map({t:_bucket(t) for t in unique_tickers})
def _passes_breakout_filters(row, BT):
    df_full=row['_df_full']; pattern=row['_pattern']; atr_series=row['_atr_series']
    breakout_idx=int(row['_breakout_idx']); end_idx=int(pattern['end_idx'])
    upper_min=BT.get('breakout_close_in_upper_pct')
    if upper_min is not None:
        bh=float(df_full['high'].iloc[breakout_idx]); bl=float(df_full['low'].iloc[breakout_idx]); bc=float(df_full['close'].iloc[breakout_idx])
        if bh>bl and (bc-bl)/(bh-bl) < upper_min: return False
    vol_mult=BT.get('breakout_volume_mult')
    if vol_mult is not None and vol_mult > vol_surge_multiplier:
        vb=float(df_full['volume'].iloc[max(0,end_idx-vol_surge_window):end_idx].mean())
        if pd.notna(vb) and vb>0 and float(df_full['volume'].iloc[breakout_idx]) < vol_mult*vb: return False
        elif not (pd.notna(vb) and vb>0): return False
    atr_min=BT.get('breakout_atr_expansion_min')
    if atr_min is not None:
        ab=atr_series.iloc[breakout_idx] if breakout_idx<len(atr_series) else np.nan
        ae=atr_series.iloc[end_idx] if end_idx<len(atr_series) else np.nan
        if pd.notna(ab) and pd.notna(ae) and ae>0:
            if (ab/ae) < atr_min: return False
        else: return False
    return True
def construct_trades(results_df, BT):
    mode=BT['entry_mode']; slip=BT['slippage_pct']; tmode=BT['target_mode']; R=BT['R_target']
    wf_buckets=_assign_wf_buckets(results_df,BT); vol_mult=BT.get('vol_surge_multiplier',1.5); vol_win=BT.get('vol_surge_window',20)
    records=[]; n_breakout_filtered=0
    for idx,row in results_df.iterrows():
        df_full=row['_df_full']; pattern=row['_pattern']; meta=row['_meta']; breakout_idx=row['_breakout_idx']
        has_buysignal=meta['buy_signal_idx'] is not None and pd.notna(meta['buy_signal_idx'])
        has_breakout=breakout_idx is not None and pd.notna(breakout_idx) and row['BreakoutDate']!='N/A'
        bs_entry_idx=None
        if has_buysignal:
            if BT.get('buysignal_entry','confirm')=='low':
                bs_entry_idx=int(meta['buy_signal_idx'])
            else:
                _tp=pattern['top_points']; _bp=pattern['bottom_points']
                bs_entry_idx=max(int(_tp[-1][0]),int(_bp[-1][0]))+int(BT.get('buysignal_entry_lag',0))
        entry_type=None; entry_idx=None
        if mode=='buysignal_preferred':
            if has_buysignal: entry_type,entry_idx='buysignal',bs_entry_idx
            elif has_breakout: entry_type,entry_idx='breakout',int(breakout_idx)
        elif mode=='breakout_only':
            if has_breakout: entry_type,entry_idx='breakout',int(breakout_idx)
        elif mode=='buysignal_only':
            if has_buysignal: entry_type,entry_idx='buysignal',bs_entry_idx
        if entry_type=='breakout' and not _passes_breakout_filters(row,BT): n_breakout_filtered+=1; continue
        if entry_type is None or entry_idx is None or entry_idx>=len(df_full): continue
        entry_raw=float(df_full['close'].iloc[entry_idx]); entry_fill=entry_raw*(1+slip)
        stop_raw=float(pattern['bottom_points'][-1][2]); stop_fill=stop_raw*(1-slip); risk=entry_fill-stop_fill
        if risk<=0: continue
        target_primary=entry_fill+R*risk; resistance=float(meta['resistance']); target_secondary=np.nan
        if tmode!='fixed_r':
            fwd=_find_next_resistance_fwd(df_full,entry_idx,resistance)
            if fwd is not None and fwd>entry_fill: target_secondary=fwd
        effective_target=target_primary if (tmode=='fixed_r' or np.isnan(target_secondary)) else min(target_primary,target_secondary)
        end_idx=pattern['end_idx']
        vol_baseline=float(df_full['volume'].iloc[max(0,end_idx-vol_win):end_idx].mean()) if end_idx<len(df_full) else 0.0
        entry_date=df_full.index[entry_idx]
        records.append({'VCP_ID':row['VCP_ID'],'Ticker':row['Ticker'],'PatternStart':row['PatternStart'],
            'PatternEnd':row['PatternEnd'],'PatternType':'rising_structure','EntryType':entry_type,
            'WF_Bucket':wf_buckets.loc[idx],'EntryIdx':entry_idx,'EntryDate':entry_date,
            'EntryRaw':round(entry_raw,4),'EntryFill':round(entry_fill,4),
            'StopRaw':round(stop_raw,4),'StopFill':round(stop_fill,4),'Risk':round(risk,4),
            'TargetPrimary':round(target_primary,4),
            'TargetSecondary':round(target_secondary,4) if not np.isnan(target_secondary) else np.nan,
            'EffectiveTarget':round(effective_target,4),'ResistanceLevel':round(resistance,4),
            'VolumeBaseline':round(vol_baseline,2),'VolSurgeMultiplier':vol_mult,'_df_full':df_full})
    trades_df=pd.DataFrame(records)
    if trades_df.empty: print('[construct_trades] No eligible trades.'); return trades_df
    trades_df=trades_df.sort_values('EntryDate').reset_index(drop=True)
    print(f"[construct_trades] {len(trades_df):,} tradeable patterns")
    if n_breakout_filtered>0: print(f"  Breakout-filter rejections: {n_breakout_filtered:,}")
    return trades_df
trades_df = construct_trades(results_df, BT)
trades_df.drop(columns=['_df_full']).head(5)

[construct_trades] 3,521 tradeable patterns


,VCP_ID,Ticker,PatternStart,PatternEnd,PatternType,EntryType,WF_Bucket,EntryIdx,EntryDate,EntryRaw,EntryFill,StopRaw,StopFill,Risk,TargetPrimary,TargetSecondary,EffectiveTarget,ResistanceLevel,VolumeBaseline,VolSurgeMultiplier
0,0,PERMNO_78987_MCHP,2020-03-13,2020-04-21,rising_structure,buysignal,IS,79,2020-04-27,42.35,42.40,37.29,37.25,5.14,68.12,44.90,44.90,41.95,9383937.50,1.50
1,1,PERMNO_76614_REGN,2020-03-05,2020-04-30,rising_structure,buysignal,IS,85,2020-05-05,574.37,574.94,507.69,507.18,67.76,913.75,578.77,578.77,574.32,1126483.80,1.50
2,2,PERMNO_79678_ATVI,2020-03-13,2020-04-29,rising_structure,buysignal,IS,85,2020-05-05,68.53,68.60,62.34,62.28,6.32,100.22,74.80,74.80,68.32,8341449.70,1.50
3,3,PERMNO_45911_SWKS,2020-03-10,2020-05-04,rising_structure,buysignal,IS,86,2020-05-06,105.90,106.01,96.73,96.63,9.37,152.87,108.29,108.29,106.85,2100303.75,1.50
4,4,PERMNO_82686_CTXS,2020-03-06,2020-04-29,rising_structure,buysignal,OOS,86,2020-05-06,147.72,147.87,139.16,139.02,8.85,192.10,155.10,155.10,152.49,2642596.80,1.50


In [74]:
def prepare_engine_price_df(df):
    out=df.copy()
    out.columns=pd.MultiIndex.from_tuples([(str(f).lower(),t) for f,t in out.columns],names=['Price','Ticker'])
    out.index=pd.DatetimeIndex(out.index); out.index.name='datetime'
    fields=list(out.columns.get_level_values('Price').unique())
    all_tickers=sorted(out.columns.get_level_values('Ticker').unique())
    full_cols=pd.MultiIndex.from_product([fields,all_tickers],names=['Price','Ticker'])
    if not out.columns.equals(full_cols):
        out=out.reindex(columns=full_cols)
    out=out.sort_index().ffill(); return out
class VCPStrategy(Strategy):
    def __init__(self,name,signals_df,config):
        self._name=name; self.signals_df=signals_df.copy().sort_values('EntryDate').reset_index(drop=True)
        self.config=config; self.exit_log=[]; self.skip_log=[]
    @property
    def name(self): return self._name
    def engine_start(self):
        self.signals_by_date=defaultdict(list)
        for _,row in self.signals_df.iterrows():
            self.signals_by_date[pd.Timestamp(row['EntryDate']).normalize()].append(row.to_dict())
        self.position_meta={}
        self._slip=float(self.config.get('slippage_pct',0.0))
        self._cap=self.config.get('max_concurrent_positions',None)
        self._max_h=int(self.config.get('max_hold_bars',120))
        self._tmode=self.config.get('target_mode','min_r_or_resistance')
        self._partial_frac=float(self.config.get('partial_frac',0.33))
        self._trail_pct=float(self.config.get('trail_pct',0.15))
    def _open_count(self): return sum(1 for t in self.cur_positions if t!='CASH')
    def _bar_ohlc(self,ticker):
        try:
            row=self.price_df.loc[self.current_bar_idx]
            o=float(row['open'][ticker]);h=float(row['high'][ticker]);l=float(row['low'][ticker]);c=float(row['close'][ticker])
            return None if any(pd.isna(x) for x in (o,h,l,c)) else (o,h,l,c)
        except: return None
    def execute_trade(self,ticker,price,quantity): return super().execute_trade(ticker,float(np.round(price,4)),quantity)
    def _bar_volume(self,ticker):
        try: v=float(self.price_df.loc[self.current_bar_idx]['volume'][ticker]); return v if pd.notna(v) else 0.0
        except: return 0.0
    def _try_exit(self,ticker):
        meta=self.position_meta.get(ticker)
        if meta is None: return False
        ohlc=self._bar_ohlc(ticker)
        if ohlc is None: return False
        o,h,l,c=ohlc; ep=meta['entry_price']
        meta['mae_pct']=min(meta['mae_pct'],(l-ep)/ep*100); meta['mfe_pct']=max(meta['mfe_pct'],(h-ep)/ep*100)
        meta['bars_held']+=1
        if self._tmode=='partial_trail' and pd.notna(meta['partial_target']) and meta['partial_target']<meta['fixed_target']-1e-9:
            return self._exit_partial_trail(ticker,meta,o,h,l,c)
        stop_raw=meta['stop']; tgt_raw=meta['target']; exit_price=None; outcome=None
        if not meta.get('target_upgraded',False):
            res=meta.get('resistance_level'); vb=meta.get('vol_baseline',0); vm=meta.get('vol_mult',1.5)
            if res and vb and vb>0 and h>=res and self._bar_volume(ticker)>=vm*vb:
                meta['target']=meta['fixed_target']; meta['target_upgraded']=True
        if l<=stop_raw: exit_price=o if h<stop_raw else stop_raw; outcome='loss'
        elif h>=tgt_raw: exit_price=o if l>tgt_raw else tgt_raw; outcome='win'
        elif meta['bars_held']>=self._max_h: exit_price=c; outcome='timeout'
        if exit_price is None: return False
        exit_price=float(max(l,min(h,exit_price)))
        if outcome!='win': exit_price*=1-self._slip; exit_price=float(max(l,min(h,exit_price)))
        qty=self.cur_positions[ticker].quantity
        self.execute_trade(ticker,exit_price,-qty)
        rmult=(exit_price-ep)/meta['risk_per_share'] if meta['risk_per_share']>0 else np.nan
        ps=qty*ep
        self.exit_log.append({'Ticker':ticker,'EntryDate':meta['entry_date'],'ExitDate':self.current_bar_idx,
            'EntryPrice':ep,'ExitPrice':exit_price,'Quantity':qty,'PositionSize':round(ps,2),
            'StopRaw':stop_raw,'EffectiveTarget':tgt_raw,'RiskPerShare':meta['risk_per_share'],
            'RMultiple':float(rmult) if rmult is not None else np.nan,
            'PnlPct':(exit_price-ep)/ep*100,'HoldingBars':meta['bars_held'],'Outcome':outcome,
            'EntryType':meta['entry_type'],'PatternType':meta['pattern_type'],
            'WF_Bucket':meta['wf_bucket'],'MAE_pct':meta['mae_pct'],'MFE_pct':meta['mfe_pct'],
            'EffRiskFrac':meta['risk_frac'],'VCP_ID':meta['vcp_id']})
        del self.position_meta[ticker]; return True
    def _try_enter(self,sig):
        ticker=sig['Ticker']
        if ticker in self.cur_positions: self.skip_log.append((self.current_bar_idx,ticker,'already_held')); return False
        if self._cap is not None and self._open_count()>=self._cap: self.skip_log.append((self.current_bar_idx,ticker,'position_cap')); return False
        ohlc=self._bar_ohlc(ticker)
        if ohlc is None: self.skip_log.append((self.current_bar_idx,ticker,'no_price_data')); return False
        o,h,l,c=ohlc
        ep_=float(sig['EntryFill']); rps=float(sig['Risk'])
        if rps<=0: self.skip_log.append((self.current_bar_idx,ticker,'zero_risk')); return False
        ep_=float(max(l,min(h,ep_)))
        nav=self.calculate_nav(inplace=False); eff_rf=float(self.config['risk_per_trade_pct'])
        shares=(nav*eff_rf)/rps
        if shares<=0: return False
        if (shares*ep_)>self.cur_positions['CASH'].quantity: self.skip_log.append((self.current_bar_idx,ticker,'insufficient_cash')); return False
        self.execute_trade(ticker,ep_,shares)
        if ticker not in self.cur_positions: self.skip_log.append((self.current_bar_idx,ticker,'zero_share_fill')); return False
        self.position_meta[ticker]={'stop':float(sig['StopRaw']),'target':float(sig['EffectiveTarget']),
            'entry_date':self.current_bar_idx,'entry_price':ep_,'risk_per_share':rps,'bars_held':0,
            'mae_pct':0.0,'mfe_pct':0.0,'entry_type':sig['EntryType'],'pattern_type':sig['PatternType'],
            'wf_bucket':sig['WF_Bucket'],'risk_frac':eff_rf,'vcp_id':sig['VCP_ID'],
            'fixed_target':float(sig.get('TargetPrimary',sig['EffectiveTarget'])),
            'partial_target':float(sig['TargetSecondary']) if pd.notna(sig.get('TargetSecondary')) else np.nan,
            'partial_done':False,'peak':ep_,'trail_stop':None,'banked_r':0.0,'init_shares':shares,
            'resistance_level':float(sig.get('ResistanceLevel',0)),'vol_baseline':float(sig.get('VolumeBaseline',0)),
            'vol_mult':float(sig.get('VolSurgeMultiplier',1.5)),'target_upgraded':False}
        return True
    def _exit_partial_trail(self,ticker,meta,o,h,l,c):
        ep=meta['entry_price'];risk=meta['risk_per_share'];stop=meta['stop']
        if not meta['partial_done']:
            if l<=stop: px=o if h<stop else stop; px=float(max(l,min(h,px)))*(1-self._slip); return self._finalize_exit(ticker,meta,float(max(l,min(h,px))),'loss',extra_r=0.0,frac=1.0)
            if h>=meta['partial_target']:
                pt=float(max(l,min(h,meta['partial_target']))); meta['banked_r']=self._partial_frac*(pt-ep)/risk
                self.execute_trade(ticker,pt,-self._partial_frac*meta['init_shares']); meta['partial_done']=True; meta['peak']=h; meta['trail_stop']=max(ep,h*(1-self._trail_pct))
            return False
        meta['peak']=max(meta['peak'],h); meta['trail_stop']=max(meta['trail_stop'],meta['peak']*(1-self._trail_pct))
        ts=meta['trail_stop']
        if l<=ts: px=o if h<ts else ts; px=float(max(l,min(h,px)))*(1-self._slip); return self._finalize_exit(ticker,meta,float(max(l,min(h,px))),'win',extra_r=meta['banked_r'],frac=(1-self._partial_frac))
        if meta['bars_held']>=self._max_h: px=float(c)*(1-self._slip); return self._finalize_exit(ticker,meta,float(max(l,min(h,px))),'timeout',extra_r=meta['banked_r'],frac=(1-self._partial_frac))
        return False
    def _finalize_exit(self,ticker,meta,eprice,outcome,extra_r,frac):
        ep=meta['entry_price'];risk=meta['risk_per_share'];qty=self.cur_positions[ticker].quantity
        self.execute_trade(ticker,eprice,-qty)
        rmult=(extra_r+frac*(eprice-ep)/risk) if risk>0 else np.nan; ps=qty*ep
        self.exit_log.append({'Ticker':ticker,'EntryDate':meta['entry_date'],'ExitDate':self.current_bar_idx,
            'EntryPrice':ep,'ExitPrice':eprice,'Quantity':qty,'PositionSize':round(ps,2),
            'StopRaw':meta['stop'],'EffectiveTarget':meta['target'],'RiskPerShare':risk,
            'RMultiple':float(rmult),'PnlPct':(eprice-ep)/ep*100,'HoldingBars':meta['bars_held'],
            'Outcome':outcome,'EntryType':meta['entry_type'],'PatternType':meta['pattern_type'],
            'WF_Bucket':meta['wf_bucket'],'MAE_pct':meta['mae_pct'],'MFE_pct':meta['mfe_pct'],
            'EffRiskFrac':meta['risk_frac'],'VCP_ID':meta['vcp_id']})
        del self.position_meta[ticker]; return True
    def on_bar(self):
        today=self.current_bar_idx
        for t in list(self.position_meta.keys()): self._try_exit(t)
        for sig in self.signals_by_date.get(pd.Timestamp(today).normalize(),[]): self._try_enter(sig)
        if today==self.price_df.index[-1]:
            for ticker in list(self.position_meta.keys()):
                ohlc=self._bar_ohlc(ticker)
                if ohlc is None: continue
                _,h,l,c=ohlc; xp=float(max(l,min(h,c)))
                meta=self.position_meta[ticker]; qty=self.cur_positions[ticker].quantity; ep=meta['entry_price']
                self.execute_trade(ticker,xp,-qty)
                _risk=meta['risk_per_share']; _frac=(1-self._partial_frac) if meta.get('partial_done') else 1.0; _banked=meta.get('banked_r',0.0)
                rmult=(_banked+_frac*(xp-ep)/_risk) if _risk>0 else np.nan; ps=qty*ep
                self.exit_log.append({'Ticker':ticker,'EntryDate':meta['entry_date'],'ExitDate':today,
                    'EntryPrice':ep,'ExitPrice':xp,'Quantity':qty,'PositionSize':round(ps,2),
                    'StopRaw':meta['stop'],'EffectiveTarget':meta['target'],'RiskPerShare':_risk,
                    'RMultiple':float(rmult),'PnlPct':(xp-ep)/ep*100,'HoldingBars':meta['bars_held'],
                    'Outcome':'eod_flat','EntryType':meta['entry_type'],'PatternType':meta['pattern_type'],
                    'WF_Bucket':meta['wf_bucket'],'MAE_pct':meta['mae_pct'],'MFE_pct':meta['mfe_pct'],
                    'EffRiskFrac':meta['risk_frac'],'VCP_ID':meta['vcp_id']})
                del self.position_meta[ticker]

# Benchmark

In [75]:
import yfinance as yf
BENCH_TICKER_KEY='BENCH_SPX'; BENCH_YF_SYMBOL='^GSPC'
def _load_benchmark_block(yf_symbol,key,target_index):
    raw=yf.download(yf_symbol,start=str(target_index[0].date()),end=str((target_index[-1]+pd.Timedelta(days=1)).date()),auto_adjust=True,progress=False)
    if raw is None or len(raw)==0: return None
    raw.columns=[c[0] if isinstance(c,tuple) else c for c in raw.columns]; raw=raw.rename(columns=str.capitalize)
    cols=['Open','High','Low','Close','Volume']; raw=raw[[c for c in cols if c in raw.columns]].copy()
    raw.index=pd.DatetimeIndex(raw.index).tz_localize(None).normalize()
    raw=raw.reindex(target_index).ffill(); raw['Volume']=raw['Volume'].fillna(0.0)
    block=pd.concat({c:raw[[c]].rename(columns={c:key}) for c in cols if c in raw.columns},axis=1)
    block.columns=block.columns.set_names(['field','Ticker']); return block
USE_BENCHMARK=True; data_for_engine=data; benchmark_strategies=[]
if USE_BENCHMARK:
    try: _bench_block=_load_benchmark_block(BENCH_YF_SYMBOL,BENCH_TICKER_KEY,data.index)
    except Exception as e: print(f'[benchmark] fetch failed ({type(e).__name__}: {e})'); _bench_block=None
    if _bench_block is not None:
        data_for_engine=pd.concat([data,_bench_block],axis=1).sort_index(axis=1)
        benchmark_strategies=[BuyAndHold('BENCH',[BENCH_TICKER_KEY])]
        print(f'[benchmark] ^GSPC spliced as {BENCH_TICKER_KEY} (buy-and-hold)')
engine_price_df=prepare_engine_price_df(data_for_engine)
print(f'engine_price_df: shape={engine_price_df.shape}, {engine_price_df.index[0].date()} → {engine_price_df.index[-1].date()}')
vcp_strategy=VCPStrategy(name='VCP',signals_df=trades_df,config=BT)
engine=BacktestEngine(price_df=engine_price_df,strategies=[vcp_strategy]+benchmark_strategies,
    initial_cash=BT['initial_cash'],risk_free_rate=BT['risk_free_rate'],margin_spread=BT['margin_spread'],
    stamp_duty=BT['commission_pct'],benchmark_strategy_name='BENCH' if benchmark_strategies else None,
    is_check_price=True,plot_trades_on_graph=False,show_all_graphs=True,
    is_plot_dict={'asset_weights':False,'realized_pnl':False,'information_ratio':True,
                  'is_num_of_assets_more_than_30':True,'random_test':False})
engine.run()

[benchmark] ^GSPC spliced as BENCH_SPX (buy-and-hold)
engine_price_df: shape=(1258, 2975), 2020-01-02 → 2024-12-31
Backtest Engine starts @12:18

Checking price_df...
price_df is valid, created immutable copy

Benchmark strategy: BENCH

Running 2 strategies...
['VCP', 'BENCH']
Running strategy 1/2: VCP
Running strategy 2/2: BENCH

Calculating performance stats for each strategy...
Backtest completed
Time taken: 0:00:36.241255

                                             VCP                BENCH
start_date                   2020-04-27 00:00:00  2020-04-27 00:00:00
end_date                     2024-12-31 00:00:00  2024-12-31 00:00:00
sharpe_ratio                                1.85                 0.75
annualized_return                           0.36                 0.17
max_drawdown                               -0.13                -0.25
drawdown_duration                            141                  512
valley_duration                               97                  282
total_ret

## VCP-Specific Analysis

In [76]:
vcp_trade_log_df=pd.DataFrame(vcp_strategy.exit_log)
if vcp_trade_log_df.empty:
    print('[VCP analysis] No completed trades.')
else:
    vcp_trade_log_df['EntryDate']=pd.to_datetime(vcp_trade_log_df['EntryDate'])
    vcp_trade_log_df['ExitDate']=pd.to_datetime(vcp_trade_log_df['ExitDate'])
    vcp_trade_log_df['EntryYear']=vcp_trade_log_df['EntryDate'].dt.year
    n_att=len(vcp_strategy.signals_df); n_done=len(vcp_trade_log_df); n_skip=len(vcp_strategy.skip_log)
    print(f'  Signals: {n_att:,}  Completed: {n_done:,}  Skipped: {n_skip:,}')
    if n_skip:
        from collections import Counter
        for reason,cnt in Counter(r for _,_,r in vcp_strategy.skip_log).most_common():
            print(f'    {reason:<22}: {cnt:>6,}')
    def _r_metrics(df,label):
        d=df.dropna(subset=['RMultiple']); n=len(d)
        if n==0: return dict(label=label,n=0)
        wins=d[d['RMultiple']>0]['RMultiple']; losses=d[d['RMultiple']<=0]['RMultiple']
        wr=len(wins)/n; aw=wins.mean() if len(wins) else 0.0; al=losses.mean() if len(losses) else 0.0
        exp=wr*aw+(1-wr)*al; pf=wins.sum()/abs(losses.sum()) if len(losses) and losses.sum()!=0 else np.inf
        rs=d['RMultiple'].std(); ah=d['HoldingBars'].mean(); tpy=252/ah if ah>0 else 0
        sharpe=(exp/rs*np.sqrt(tpy)) if rs>0 else np.nan
        return dict(label=label,n=n,win_rate=round(wr,4),avg_win_r=round(aw,3),avg_loss_r=round(al,3),
            expectancy=round(exp,4),profit_factor=round(pf,3),total_r=round(d['RMultiple'].sum(),2),
            avg_hold_bars=round(ah,1),sharpe_r=round(sharpe,3),
            avg_mae_pct=round(d['MAE_pct'].mean(),3),avg_mfe_pct=round(d['MFE_pct'].mean(),3))
    rows=[_r_metrics(vcp_trade_log_df,'ALL TRADES')]
    for bkt in ['IS','OOS']: rows.append(_r_metrics(vcp_trade_log_df[vcp_trade_log_df['WF_Bucket']==bkt],f'WF={bkt}'))
    for et in vcp_trade_log_df['EntryType'].unique(): rows.append(_r_metrics(vcp_trade_log_df[vcp_trade_log_df['EntryType']==et],f'EntryType={et}'))
    rpt=pd.DataFrame(rows).set_index('label')
    fmt=dict(n='{:,.0f}',win_rate='{:.1%}',avg_win_r='{:+.2f}R',avg_loss_r='{:+.2f}R',
             expectancy='{:+.4f}R',profit_factor='{:.2f}x',total_r='{:+.1f}R',avg_hold_bars='{:.0f}d',
             sharpe_r='{:.3f}',avg_mae_pct='{:+.2f}%',avg_mfe_pct='{:+.2f}%')
    disp=rpt.copy()
    for col,f in fmt.items():
        if col in disp.columns: disp[col]=disp[col].apply(lambda v: f.format(v) if pd.notna(v) else '—')
    print(); print('──── R-multiple segment report────'); display(disp)

  Signals: 3,521  Completed: 756  Skipped: 2,765
    insufficient_cash     :  2,721
    already_held          :     44

──── R-multiple segment report────


,n,win_rate,avg_win_r,avg_loss_r,expectancy,profit_factor,total_r,avg_hold_bars,sharpe_r,avg_mae_pct,avg_mfe_pct
label,,,,,,,,,,,
ALL TRADES,756,75.4%,+0.61R,-0.99R,+0.2158R,1.89x,+163.1R,8d,1.355,-2.77%,+3.20%
WF=IS,560,75.9%,+0.64R,-0.98R,+0.2498R,2.05x,+139.9R,8d,1.473,-2.77%,+3.34%
WF=OOS,196,74.0%,+0.51R,-1.00R,+0.1184R,1.45x,+23.2R,7d,0.923,-2.76%,+2.80%
EntryType=buysignal,756,75.4%,+0.61R,-0.99R,+0.2158R,1.89x,+163.1R,8d,1.355,-2.77%,+3.20%


In [77]:
from pathlib import Path
def trade_export(trade_log_df,output_path="vcp_trade_export.csv",results_df=None,trades_df=None):
    if trade_log_df.empty: print('[trade_export] No trades.'); return trade_log_df
    export=trade_log_df.copy()
    if 'EntryDate' in export.columns: export['EntryDate']=pd.to_datetime(export['EntryDate'])
    if 'ExitDate' in export.columns: export['ExitDate']=pd.to_datetime(export['ExitDate'])
    if 'EntryDate' in export.columns and 'ExitDate' in export.columns:
        export['HoldingPeriodDays']=(export['ExitDate']-export['EntryDate']).dt.days
    if 'Quantity' in export.columns: export['PnL_Dollar']=export['Quantity']*(export['ExitPrice']-export['EntryPrice'])
    else: export['PnL_Dollar']=np.nan
    if 'Quantity' in export.columns: export['PositionSize']=export['Quantity']*export['EntryPrice']
    if results_df is not None and 'VCP_ID' in export.columns and 'VCP_ID' in results_df.columns:
        mc=[c for c in ['VCP_ID','PatternStart','PatternEnd','LastSwingPct','BuySignalDate','BreakoutDate'] if c in results_df.columns]
        if mc: export=export.merge(results_df[mc],on='VCP_ID',how='left')
    if trades_df is not None and 'VCP_ID' in export.columns and 'VCP_ID' in trades_df.columns:
        tc=[c for c in ['VCP_ID','EntryFill','EntryRaw','StopFill','TargetPrimary','TargetSecondary',
                        'EffectiveTarget','ResistanceLevel','VolumeBaseline','VolSurgeMultiplier'] if c in trades_df.columns]
        if tc: export=export.merge(trades_df[tc],on='VCP_ID',how='left')
    preferred=['VCP_ID','Ticker','EntryType','WF_Bucket','PatternStart','PatternEnd','PatternType',
        'LastSwingPct','BuySignalDate','BreakoutDate','EntryDate','ExitDate','HoldingPeriodDays','HoldingBars',
        'Quantity','EntryPrice','ExitPrice','EntryRaw','EntryFill','StopRaw','StopFill','RiskPerShare',
        'EffectiveTarget','TargetPrimary','TargetSecondary','ResistanceLevel','PositionSize','PnL_Pct',
        'PnL_Dollar','RMultiple','Outcome','MAE_pct','MFE_pct','EffRiskFrac','VolumeBaseline','VolSurgeMultiplier']
    avail=[c for c in preferred if c in export.columns]
    export=export[avail+[c for c in export.columns if c not in avail]]
    export.to_csv(output_path,index=False)
    print(f'[trade_export] {len(export)} trades written to {output_path}')
    print(f'  Win rate: {len(export[export["RMultiple"]>0])/len(export):.1%}  Avg R: {export["RMultiple"].mean():+.3f}  Total R: {export["RMultiple"].sum():+.1f}')
    return export
if not vcp_trade_log_df.empty:
    vcp_trade_export=trade_export(vcp_trade_log_df,output_path='vcp_trade_log.csv',
        results_df=results_df if 'results_df' in dir() or 'results_df' in builtins.globals() else None,
        trades_df=trades_df if 'trades_df' in dir() or 'trades_df' in builtins.globals() else None)
else: print('[trade_export] vcp_trade_log_df is empty.')

[trade_export] 756 trades written to vcp_trade_log.csv
  Win rate: 75.4%  Avg R: +0.216  Total R: +163.1


## Buy Signal Verification

Re-runs VCP detection on the **causal window** (pattern start → buy signal date) for a
given PERMNO key. This lets you visually confirm that the correct local tops and bottoms
were identified using only the data available at signal time.

DC pivot lines (horizontal dashed from top extreme to next bottom extreme) are drawn
to make the structure detection transparent.

In [81]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

def isBuySignal_check(permno: str):
    import builtins
    _results_df = builtins.globals().get('results_df')
    if _results_df is None:
        print('[isBuySignal_check] results_df not available.'); return
    results_df = _results_df
    matches = results_df[results_df['Ticker'] == permno]
    if matches.empty:
        print(f'[isBuySignal_check] No signals found for {permno}.'); return
    print(f'[isBuySignal_check] {permno}: {len(matches)} signal(s) detected')
    for _, row in matches.iterrows():
        vcp_id = row['VCP_ID']
        pattern_start = pd.Timestamp(row['PatternStart']).normalize()
        signal_date   = pd.Timestamp(row['BuySignalDate']).normalize()
        last_swing_pct = row['LastSwingPct']
        df_full = row['_df_full']
        exit_date = None
        if not vcp_trade_log_df.empty:
            trade_row = vcp_trade_log_df[vcp_trade_log_df['VCP_ID'] == vcp_id]
            if not trade_row.empty:
                exit_date = pd.Timestamp(trade_row.iloc[0]['ExitDate']).normalize()
        df_window = df_full.loc[pattern_start:signal_date].copy()
        if len(df_window) < 20:
            print(f'  VCP#{vcp_id}: insufficient data in causal window, skipping'); continue
        cnp=df_window['close'].to_numpy(); hnp=df_window['high'].to_numpy(); lnp=df_window['low'].to_numpy()
        atr_s=compute_atr(df_window, dc_atr_window)
        sigma_s=(dc_atr_multiplier*atr_s/df_window['close']).clip(dc_sigma_floor, dc_sigma_ceiling).to_numpy()
        tops, bottoms = directional_change(cnp, hnp, lnp, sigma_s)
        chart_end = exit_date if exit_date is not None else signal_date + pd.Timedelta(days=120)
        df_chart = df_full.loc[pattern_start:chart_end].copy()
        title = f'{permno}  |  VCP#{vcp_id}  |  Swing={last_swing_pct:.2f}%'
        if exit_date is not None and not trade_row.empty:
            tr = trade_row.iloc[0]
            title += f'  |  {tr["Outcome"]}  |  R={tr["RMultiple"]:+.2f}  |  PnL={tr["PnlPct"]:+.2f}%'
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                            row_heights=[0.7, 0.3], subplot_titles=[title, 'Volume'])
        fig.add_trace(go.Candlestick(x=df_chart.index, open=df_chart['open'], high=df_chart['high'],
            low=df_chart['low'], close=df_chart['close'], name=permno, showlegend=False,
            increasing_line_color='#4f98a3', decreasing_line_color='#e94560'), row=1, col=1)
        # Use add_shape for vertical line to avoid pandas Timestamp arithmetic bug in add_vline
        fig.add_shape(type='line', x0=signal_date, x1=signal_date, y0=0, y1=1, yref='paper',
                      line=dict(color='#ffd600', width=1, dash='dash'), row=1, col=1)
        fig.add_annotation(x=signal_date, y=1, yref='paper', text='Signal', showarrow=False,
                          font=dict(color='#ffd600'), xanchor='left', row=1, col=1)
        pivot_index = df_window.index
        if tops:
            fig.add_trace(go.Scatter(x=[pivot_index[t[1]] for t in tops], y=[t[2] for t in tops],
                mode='markers', marker=dict(color='#ffd600',size=8,symbol='triangle-down',
                    line=dict(color='black',width=0.5)),
                name='DC Top', showlegend=False), row=1, col=1)
        if bottoms:
            fig.add_trace(go.Scatter(x=[pivot_index[b[1]] for b in bottoms], y=[b[2] for b in bottoms],
                mode='markers', marker=dict(color='#00ff88',size=8,symbol='triangle-up',
                    line=dict(color='black',width=0.5)),
                name='DC Bottom', showlegend=False), row=1, col=1)
        for ti in range(min(len(tops), len(bottoms))):
            t_idx=tops[ti][1]; b_idx=bottoms[ti][1]
            t_d=pivot_index[t_idx]; t_p=tops[ti][2]
            b_d=pivot_index[b_idx]; b_p=bottoms[ti][2]
            if b_d>=t_d:
                fig.add_trace(go.Scatter(x=[t_d,b_d],y=[t_p,t_p],mode='lines',
                    line=dict(color='#ffd600',width=1,dash='dot'),showlegend=False,hoverinfo='skip'),row=1,col=1)
                fig.add_trace(go.Scatter(x=[b_d,b_d],y=[t_p,b_p],mode='lines',
                    line=dict(color='#00ff88',width=1,dash='dot'),showlegend=False,hoverinfo='skip'),row=1,col=1)
        fig.add_vrect(x0=pattern_start, x1=signal_date, fillcolor='#4f98a3', opacity=0.06,
                      annotation_text=f'VCP (swing={last_swing_pct:.1f}%)',
                      annotation_position='top left', line_width=0, row=1, col=1)
        buy_price = row['_meta']['buy_signal_price']
        if signal_date in df_chart.index:
            fig.add_trace(go.Scatter(x=[signal_date], y=[buy_price], mode='markers',
                marker=dict(color='#00ff88',size=14,symbol='diamond',line=dict(color='white',width=1.5)),
                showlegend=False, hovertemplate=f'BUY {signal_date.date()}<br>${buy_price:.2f}'),row=1,col=1)
        if exit_date is not None and not trade_row.empty:
            tr=trade_row.iloc[0]; ep_=float(tr['ExitPrice']); outcome=tr['Outcome']
            ec='#4fc3f7' if outcome=='win' else ('#e94560' if outcome=='loss' else '#aaaaaa')
            if exit_date in df_chart.index:
                fig.add_trace(go.Scatter(x=[exit_date], y=[ep_],mode='markers',
                    marker=dict(color=ec,size=14,symbol='diamond',line=dict(color='white',width=1.5)),
                    showlegend=False, hovertemplate=f'{outcome.upper()} {exit_date.date()}<br>${ep_:.2f}'),row=1,col=1)
        fig.add_trace(go.Bar(x=df_chart.index, y=df_chart['volume'],
            marker_color='#4f98a3', opacity=0.6, showlegend=False), row=2, col=1)
        fig.update_layout(height=500, template='plotly_dark', paper_bgcolor='#1a1a2e',
            plot_bgcolor='#16213e', margin=dict(l=30,r=30,t=50,b=30), xaxis_rangeslider_visible=False)
        fig.update_xaxes(rangebreaks=[dict(bounds=['sat','mon'])], showgrid=True, gridcolor='#2a2a4a', gridwidth=0.4)
        fig.update_yaxes(showgrid=True, gridcolor='#2a2a4a', gridwidth=0.4)
        fig.show(); print()

isBuySignal_check("PERMNO_12872_MPC")

[isBuySignal_check] PERMNO_12872_MPC: 12 signal(s) detected
